# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/dev-hashh/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 125 (delta 40), reused 84 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 1.88 MiB | 6.89 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/flyrank-ml-internship-starter


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [7]:
import duckdb
from google.colab import userdata


HF_TOKEN = userdata.get("FLYRANK_AI")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

result = con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

result.show()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘



## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily search and engagement performance of a single content page for a single client on one report date.

The analysis uses the warehouse data for March 2026 (month='2026-03'). This mid-panel month is used to avoid developing logic on the final month (June 2026), which is reserved as a sealed test period.

In [20]:
result = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03';
""")

result.show()

┌───────────┬────────────┬────────────┐
│ row_count │ start_date │  end_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
result = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS pages,
    COUNT(DISTINCT report_date) AS dates
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03';
""")

result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────┬────────┬───────┐
│ total_rows │ clients │ pages  │ dates │
│   int64    │  int64  │ int64  │ int64 │
├────────────┼─────────┼────────┼───────┤
│    9841378 │      55 │ 331437 │    31 │
└────────────┴─────────┴────────┴───────┘



In [9]:
result = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS cnt
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
GROUP BY 1,2,3
HAVING COUNT(*) > 1;
""")

result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────┐
│ client_hash_id │ content_hash_id │ report_date │  cnt  │
│    varchar     │     varchar     │    date     │ int64 │
├────────────────┴─────────────────┴─────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



In [10]:
result = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03';
""")

result.show()

┌───────────┬────────────┬────────────┐
│ row_count │ start_date │  end_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



This confirms the dataset slice covers March 2026.

In [11]:
result = con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
  AND ga4_data_available IS TRUE;
""")

result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│         413966 │
└────────────────┘



In [12]:
result = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS available_rows,
    ROUND(
        100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS availability_pct
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03';
""")

result.show()


┌────────────┬────────────────┬──────────────────┐
│ total_rows │ available_rows │ availability_pct │
│   int64    │     int128     │      double      │
├────────────┼────────────────┼──────────────────┤
│    9841378 │         413966 │             4.21 │
└────────────┴────────────────┴──────────────────┘



## **FIVE FEATURES:**

In [17]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')

""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [19]:
features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_engaged_sessions,
    scroll_events
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
  AND ga4_data_available IS TRUE
LIMIT 1000;
""")

features.df().head()

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,scroll_events
0,client_65de48885f4ef01b,content_09be8cc7fcb222af,2026-03-01,0,0,NaN,0,0
1,client_65de48885f4ef01b,content_851afac9fe13612e,2026-03-01,0,0,NaN,0,0
2,client_65de48885f4ef01b,content_cee6c6fc8c51af14,2026-03-01,0,0,NaN,0,0
3,client_65de48885f4ef01b,content_5e120e972f11f833,2026-03-01,0,0,NaN,0,0
4,client_65de48885f4ef01b,content_16a7291bb6ecaebe,2026-03-01,0,0,NaN,0,0


## Five Selected Features

### Feature 1: `gsc_impressions`
**Available when?**  
Knowable at the decision moment because search impressions are historical observations already available before deciding whether a page should be refreshed.

---

### Feature 2: `gsc_clicks`
**Available when?**  
Knowable at the decision moment because historical click data has already been collected before the refresh decision.

---

### Feature 3: `gsc_avg_position`
**Available when?**  
Knowable at the decision moment because the average search position is computed from historical Google Search Console data available before making the decision.

---

### Feature 4: `ga4_engaged_sessions`
**Available when?**  
Knowable at the decision moment because user engagement metrics are collected before deciding whether a page should be refreshed.

---

### Feature 5: `scroll_events`
**Available when?**  
Knowable at the decision moment because scroll events represent historical user interactions with the page and are observed before the refresh decision.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.